## Generating Structured Output

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv pydantic

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=anthropic_api_key)

### Generating Structured Output using Raw JSON Schema Configuration

In [ ]:
user_prompt = """
Extract the following interviewee's info into structured JSON.

Interviewee Name: John Doe
Age: 30
Email: john.doe@example.com
Company: Example Corp
The interviewee has 5 years of experience in software development and is skilled in Python, JavaScript, and cloud technologies.
He is not serving notice period and is available to join immediately.
"""

response = client.messages.create(
    model=claude_model,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "age": {"type": "integer"},
                    "email": {"type": "string"},
                    "company": {"type": "string"},
                    "experience_years": {"type": "integer"},
                    "skills": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "is_serving_notice_period": {"type": "boolean"},
                },
                "required": ["name", "age", "email", "company", "experience_years", "skills", "is_serving_notice_period"],
                "additionalProperties": False
            }
        }
    }
)

for block in response.content:
    if block.type == "text":
        print(block.text)

### Using Pydantic Model to Generate Structured Output

In [ ]:
from pydantic import BaseModel


class IntervieweeInfo(BaseModel):
    name: str
    age: int
    email: str
    company: str
    experience_years: int
    skills: list[str]
    is_serving_notice_period: bool


response = client.messages.parse(
    model=claude_model,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    output_format=IntervieweeInfo
)

interviewee_info = response.parsed_output
print("Interviewee Name: ", interviewee_info.name)
print("Age: ", interviewee_info.age)
print("Email: ", interviewee_info.email)
print("Company: ", interviewee_info.company)
print("Experience (years): ", interviewee_info.experience_years)
print("Skills: ", interviewee_info.skills)
print("Serving Notice Period: ", interviewee_info.is_serving_notice_period)